# Nhiệm vụ 8: Phân tích Hồ sơ Cụm (Cluster Profiling)

**Mục đích tổng quát:** Khắc họa và nhận diện rõ nét tính chất (hồ sơ) của 2 cụm cổ phiếu (K=2) dựa trên thuật toán Ward, thông qua lăng kính của 8 yếu tố cốt lõi: Động lượng (Momentum), Rủi ro (Vol, MDD, Beta), và Thanh khoản (Liquidity).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os
from collections import defaultdict

ward_dir = '../artifacts/m2-task5-ward-v1'
eval_dir = '../artifacts/m2-evaluation-ward'
os.makedirs(eval_dir, exist_ok=True)

FEATURES = ["beta_126", "liquidity_21", "mdd_126", "mom_126", "mom_21", "mom_252", "mom_63", "vol_63"]

# 1. Đọc dữ liệu Assignments và Profiles
assignments = []
with open(os.path.join(ward_dir, 'assignments.jsonl'), 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip(): assignments.append(json.loads(line))

by_date = defaultdict(dict)
for row in assignments:
    by_date[row['snapshot_date']][row['security_id']] = row
days = sorted(by_date.keys())

profiles = []
with open(os.path.join(ward_dir, 'profiles.jsonl'), 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip(): profiles.append(json.loads(line))
profile_index = {(p['snapshot_date'], p['raw_cluster_id']): p for p in profiles}

# 2. Tổng hợp Hồ sơ Cụm (Profiles) và Khôi phục giá trị Scaled
profile_rows = []
scaled_rows = []

for day in days:
    members = by_date[day]
    model_path = os.path.join(ward_dir, 'models', f'{day}.json')
    with open(model_path, 'r', encoding='utf-8') as f:
        model = json.load(f)
        
    for raw_id in (0, 1):
        profile = profile_index.get((day, raw_id))
        if not profile: continue
            
        aligned = profile["aligned_cluster_id"]
        size = profile["size"]
        
        flat = {
            "method": "ward", "snapshot_date": day, "k": 2, 
            "raw_cluster_id": raw_id, "aligned_cluster_id": aligned, 
            "size": size, "share": size / len(members), 
            "n_eligible": len(members), "economic_rank": profile.get("economic_rank")
        }
        flat.update(profile["centroid"])
        profile_rows.append(flat)
        
        for column, feature in enumerate(FEATURES):
            scale = model["scaler"][feature]
            raw_mean = profile["centroid"][feature]
            transformed = (raw_mean - scale["center"]) / scale["scale"] if scale["scale"] else 0
            scaled_rows.append({
                "snapshot_date": day, "aligned_cluster_id": aligned, "feature": feature,
                "raw_mean": raw_mean, "scaled_mean": transformed,
                "scaler_center": scale["center"], "scaler_scale": scale["scale"]
            })

df_profiles = pd.DataFrame(profile_rows)
df_profiles.to_csv(os.path.join(eval_dir, 'cluster_profiles.csv'), index=False)

df_members = pd.DataFrame(assignments)
df_members['method'] = 'ward'
df_members.to_csv(os.path.join(eval_dir, 'cluster_profile_members.csv'), index=False)

df_scaled = pd.DataFrame(scaled_rows)
df_scaled.to_csv(os.path.join(eval_dir, 'profile_scaler_reference.csv'), index=False)

print("Đã sinh ra các file evaluation cho Task 8:\n- cluster_profiles.csv\n- cluster_profile_members.csv\n- profile_scaler_reference.csv\n")
df = df_profiles.rename(columns={'aligned_cluster_id': 'cluster'})

### Phần 8.1 — Tổng hợp feature theo cụm

**Mục đích:**
Xác định đặc điểm điển hình. Tổng hợp các feature đại diện của từng cluster.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def show_features_for_month(target_date):
    df_target = df[df['snapshot_date'] == target_date].copy()
    print(f'Tổng hợp features của các cụm tại snapshot: {target_date}\n')
    display(df_target[['cluster', 'size'] + FEATURES])

dropdown = widgets.Dropdown(options=days, value=days[-1], description='Chọn Tháng:')
widgets.interact(show_features_for_month, target_date=dropdown);

**Đánh giá Đặc điểm Điển hình:**
- Bảng trên liệt kê trung bình các giá trị đặc trưng của từng cụm (dạng Raw/Gốc). 
- Tại K=2, có sự khác biệt rất lớn về mặt quy mô số lượng mã (`size`). Một cụm thường rất lớn (đa số thị trường) và một cụm rất nhỏ (nhóm thiểu số).
- Các chỉ số động lượng (momentum) và rủi ro (vol, mdd) cho thấy các giá trị bình quân của cụm lớn rất sát với mức trung bình chung của toàn thị trường, trong khi cụm nhỏ mang các giá trị cực đoan (rất thấp hoặc rất cao).

### Phần 8.2 — So sánh giữa các cụm

**Mục đích:**
Xác định feature nào làm các cụm khác nhau. So sánh động lượng, rủi ro, beta và thanh khoản giữa các cụm.

**Lưu ý về Biểu đồ Radar:**
Biểu đồ dưới đây sử dụng **Robust Z-Score** (thang đo chuẩn hóa của model). Đường đứt nét màu xám (y=0) đại diện cho **Trung vị Thị trường (Market Median)**. 
- Mọi điểm nằm **bên ngoài** đường 0 thể hiện giá trị cao hơn trung bình thị trường.
- Mọi điểm nằm **bên trong** đường 0 thể hiện giá trị thấp hơn trung bình thị trường.
- Các giá trị cực đoan được giới hạn (clip) trong khoảng [-3, 3] để biểu đồ không bị bóp méo.

In [ ]:
def plot_radar_for_month(target_date):
    if df.empty: return
    df_target = df[df['snapshot_date'] == target_date].copy()
    if df_target.empty: return
        
    features = ['mom_21', 'mom_63', 'mom_126', 'mom_252', 'vol_63', 'mdd_126', 'beta_126', 'liquidity_21']
    labels = np.array(features)
    num_vars = len(labels)
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1]
    
    fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    
    df_s = df_scaled[df_scaled['snapshot_date'] == target_date]
    
    # Vẽ đường Market Median (y = 0)
    ax.set_ylim(-3, 3)
    ax.plot(np.linspace(0, 2*np.pi, 100), np.zeros(100), color='gray', linestyle='--', linewidth=2, alpha=0.7, label="Market Median (0)")
    
    # Ẩn nhãn trục y, giữ trục lưới
    ax.set_yticks([-2, -1, 1, 2])
    ax.set_yticklabels([])
    ax.grid(True, color='#DDDDDD', linestyle='-', linewidth=1)
    
    for idx, row_orig in df_target.iterrows():
        cluster_id = row_orig['cluster']
        c_size = row_orig['size']
        
        vals = []
        for f in features:
            s_val = df_s[(df_s['aligned_cluster_id'] == cluster_id) & (df_s['feature'] == f)]['scaled_mean'].values
            val = s_val[0] if len(s_val) > 0 else 0
            val = np.clip(val, -3, 3) # Clip outlier
            vals.append(val)
            
        vals += vals[:1]
        
        ax.plot(angles, vals, color=colors[cluster_id % len(colors)], linewidth=3, label=f"Cluster {cluster_id} (n={c_size})")
        ax.fill(angles, vals, color=colors[cluster_id % len(colors)], alpha=0.15)
    
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=12, fontweight='bold')
    
    plt.title(f'Radar Chart: Hồ sơ Đặc trưng các Cụm (Ward K=2)\nTháng: {target_date}', y=1.1, fontsize=15, fontweight='bold')
    plt.legend(loc='upper right', bbox_to_anchor=(1.25, 1.15), fontsize=12)
    plt.tight_layout()
    plt.show()

import ipywidgets as widgets
available_dates = sorted(df['snapshot_date'].unique())
dropdown_radar = widgets.Dropdown(options=available_dates, value=available_dates[-1], description='Chọn Tháng:')
widgets.interact(plot_radar_for_month, target_date=dropdown_radar);


**Phân tích So sánh & Đặc điểm tương quan:**
Nhờ vào hệ tọa độ Chuẩn hóa, sự khác biệt giữa hai cụm hiện lên cực kỳ tương phản và trực quan:
1. **Cụm đám đông (Đường bám sát tâm 0):** Cụm này chứa gần như toàn bộ thị trường. Vì số lượng mã quá áp đảo, giá trị trung bình của cụm này luôn nằm dính chặt vào đường "Market Median". 
2. **Cụm thiểu số (Đường ziczac sắc nhọn):** Chứa vài chục (hoặc vài) mã có tính chất cực đoan, dị biệt hoàn toàn so với thị trường.
   - Khi thị trường biến động mạnh, cụm này sẽ phóng vọt ra ngoài (hoặc đâm sâu vào trong) ở các trục động lượng (momentum) hoặc rủi ro.
   - Thể hiện sự "phân mảnh" (fragmentation) của Ward, khi nó quyết định tách riêng các cổ phiếu mang tính chất Outlier (thanh khoản siêu lớn, hoặc bị đình chỉ, hoặc rơi tự do) thành một nhóm riêng.

Sự khác biệt lớn nhất giữa hai cụm chính là sự đối lập gay gắt giữa một nhóm **Bình thường (Normal)** và một nhóm **Dị biệt (Outliers)**.

### Phần 8.3 — Giới hạn diễn giải

**Mục đích:**
Không biến kết quả M2 thành khuyến nghị đầu tư. Không kết luận cluster nào là cluster nên mua.

**Giới hạn diễn giải phân tích:**
- Kết quả từ Ward clustering (K=2) phản ánh trạng thái tĩnh toán học của thị trường tại thời điểm cắt (snapshot) dựa trên không gian khoảng cách đã chuẩn hoá.
- Bất kỳ quan sát nào về "Cụm này có động lượng mạnh" hay "Cụm kia có rủi ro thấp" **KHÔNG ĐƯỢC PHÉP** biến thành khuyến nghị mua/bán cổ phiếu.
- Việc một cụm thiểu số mang các giá trị cực cao không có nghĩa là nó sẽ mang lại Alpha. Thực tế, đó có thể chỉ là các mã có thanh khoản dị biệt, biến động bất thường, hoặc chịu hiệu ứng sống sót (survivorship bias) trong tháng đó.
- Nhắc lại: Mục đích của M2 chỉ là khám phá đặc tính vi mô của các cấu trúc nhóm (Unsupervised Learning). Không kết luận cụm nào là cụm nên mua khi chưa có backtest danh mục thực tế.